# Dataset Verification and Manifest Generation

Scans the raw FFE directory tree, checks image integrity, and writes a labeled manifest CSV consumed by every downstream notebook.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Pronnnnnnn/fish-freshness-mtl.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

Mounted at /content/drive
Cloning into '/content/repo'...
remote: Enumerating objects: 67, done.
remote: Counting objects: 100% (67/67), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 67 (delta 27), reused 67 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (67/67), 83.67 KiB | 1.82 MiB/s, done.
Resolving deltas: 100% (27/27), done.
/content/repo
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 73.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
DATASET_ZIP = '/content/drive/MyDrive/fish-freshness-mtl/8_fish_3_freshness.zip'
DATASET_ROOT = '/content/data/8_fish_3_freshness'

import os
os.makedirs('/content/data', exist_ok=True)
!unzip -q "$DATASET_ZIP" -d /content/data
!find "$DATASET_ROOT" -maxdepth 1 -type d | wc -l

25


In [5]:
import sys
sys.path.append('/content/repo/04_Src')

from manifest_utils import build_manifest, summarize_class_distribution

df = build_manifest(DATASET_ROOT)
print(f'total images: {len(df)}')
df.head()

total images: 4390


,filepath,species,freshness,species_idx,freshness_idx,combined_class
0,Chanos Chanos - Fresh/IMG_20191002_062711.jpg,Chanos Chanos,Fresh,0,1,Chanos Chanos | Fresh
1,Chanos Chanos - Fresh/IMG_20191002_062722.jpg,Chanos Chanos,Fresh,0,1,Chanos Chanos | Fresh
2,Chanos Chanos - Fresh/IMG_20191002_062735.jpg,Chanos Chanos,Fresh,0,1,Chanos Chanos | Fresh
3,Chanos Chanos - Fresh/IMG_20191002_062742.jpg,Chanos Chanos,Fresh,0,1,Chanos Chanos | Fresh
4,Chanos Chanos - Fresh/IMG_20191002_062803.jpg,Chanos Chanos,Fresh,0,1,Chanos Chanos | Fresh


In [6]:
summarize_class_distribution(df)

freshness,Highly Fresh,Fresh,Not Fresh,Total
species,,,,
Chanos Chanos,168,162,170,500
Eleutheronema Tetradactylum,80,80,80,240
Johnius Trachycephalus,80,80,80,240
Nibea Albiflora,173,125,121,419
Oreochromis Mossambicus,289,174,162,625
Oreochromis Niloticus,328,231,246,805
Rastrelliger Faughni,336,216,217,769
Upeneus Moluccensis,310,252,230,792
Total,1764,1320,1306,4390


In [7]:
EXPECTED_TOTAL = 4390
assert len(df) == EXPECTED_TOTAL, f'expected {EXPECTED_TOTAL} images, found {len(df)}'
print('image count matches expected total')

image count matches expected total


## Integrity check: corrupt and duplicate files

In [8]:
import hashlib
from pathlib import Path
from PIL import Image, UnidentifiedImageError

corrupt = []
hashes = {}
duplicates = []

for rel_path in df['filepath']:
    path = Path(DATASET_ROOT) / rel_path
    try:
        with Image.open(path) as im:
            im.verify()
    except (UnidentifiedImageError, OSError):
        corrupt.append(rel_path)
        continue
    with open(path, 'rb') as f:
        h = hashlib.md5(f.read()).hexdigest()
    if h in hashes:
        duplicates.append((rel_path, hashes[h]))
    else:
        hashes[h] = rel_path

print(f'corrupt files: {len(corrupt)}')
print(f'duplicate files: {len(duplicates)}')

corrupt files: 0
duplicate files: 0


In [10]:
OUT_PATH = '/content/repo/02_Manifests/manifest_full.csv'
df.to_csv(OUT_PATH, index=False)
print('saved to', OUT_PATH)

saved to /content/repo/02_Manifests/manifest_full.csv
